In [ ]:
from sklearn.linear_model import LinearRegression,LogisticRegression
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
df=pd.read_csv('iris_extended.csv')

encoder=LabelEncoder()
df['Species']=encoder.fit_transform(df['Species'])
features=df.drop('Species',axis=1)
target=df['Species']
X_train, X_test, y_train, y_test = train_test_split(features,target, test_size=0.35, random_state=42)

scaler = MinMaxScaler()
x_train=scaler.fit_transform(X_train)
x_test=scaler.transform(X_test)

In [12]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def ann_numpy_classifier(X_train, y_train, X_test, y_test, epochs=1000, hidden_size=8, learning_rate=0.01):
    np.random.seed(42)

    input_size = X_train.shape[1]
    output_size = y_train.shape[1]

    # Weight and bias initialization
    W1 = np.random.randn(input_size, hidden_size)
    b1 = np.zeros((1, hidden_size))

    W2 = np.random.randn(hidden_size, output_size)
    b2 = np.zeros((1, output_size))

    # Training loop
    for epoch in range(epochs):
        # Forward pass
        Z1 = np.dot(X_train, W1) + b1
        A1 = sigmoid(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        # Loss (cross-entropy)
        loss = -np.mean(np.sum(y_train * np.log(A2 + 1e-8), axis=1))

        # Backward pass
        dZ2 = A2 - y_train
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * sigmoid_derivative(Z1)
        dW1 = np.dot(X_train.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    # ---- Prediction ----
    def predict(X):
        A1 = sigmoid(np.dot(X, W1) + b1)
        A2 = softmax(np.dot(A1, W2) + b2)
        return np.argmax(A2, axis=1)

    y_pred = predict(X_test)
    y_true = np.argmax(y_test, axis=1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    accuracy = np.mean(y_pred == y_true)
    print(f"\nTest Accuracy: {accuracy:.4f}")

    return y_pred, (W1, b1, W2, b2)


In [13]:
from sklearn.preprocessing import OneHotEncoder

# Reshape for encoder
y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.values.reshape(-1, 1)

# Use compatible argument name
encoder_ohe = OneHotEncoder(sparse_output=False)
y_train_encoded = encoder_ohe.fit_transform(y_train_reshaped)
y_test_encoded = encoder_ohe.transform(y_test_reshaped)

# Call the classifier
ann_numpy_classifier(x_train, y_train_encoded, x_test, y_test_encoded, epochs=50, hidden_size=4, learning_rate=0.01)

Epoch 0, Loss: 1.2758
Epoch 10, Loss: 0.9926
Epoch 20, Loss: 0.8120
Epoch 30, Loss: 0.6636
Epoch 40, Loss: 0.5644
Epoch 49, Loss: 0.5050

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      0.30      0.46        20
           2       0.56      1.00      0.72        18

    accuracy                           0.75        57
   macro avg       0.85      0.77      0.73        57
weighted avg       0.86      0.75      0.72        57

Confusion Matrix:
[[19  0  0]
 [ 0  6 14]
 [ 0  0 18]]

Test Accuracy: 0.7544


(array([2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 2, 0, 2, 0, 0, 2, 2, 1, 0, 2, 0,
        0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1, 2, 0, 2, 0, 0, 1, 2, 0, 2, 1, 2,
        2, 0, 1, 0, 2, 1, 2, 0, 2, 2, 0, 0, 2]),
 (array([[ 0.0147545 , -0.06317219,  0.39570319,  2.20322204],
         [-0.91350016, -0.14523281,  1.60965664,  1.47123382],
         [ 1.0375405 ,  0.41497129, -0.79152009, -0.88126462],
         [-1.43883267, -1.7271108 , -1.66571001,  0.83808902],
         [-2.71850521,  0.52931124, -1.05830785,  0.31639529]]),
  array([[ 1.06294818, -0.15739923,  0.3218739 , -0.73051868]]),
  array([[ 3.58298306, -0.62820687, -1.64737551],
         [-0.23548882, -0.78784147, -0.83487804],
         [-0.26298114,  0.54784288, -1.660796  ],
         [-1.46597358, -0.15273066,  2.57758206]]),
  array([[-0.50434572,  0.55378119, -0.04943547]])))

In [15]:
model = Sequential([
    Dense(10, activation='relu', input_shape=(x_train.shape[1],)),  # Hidden layer
    Dense(y_train_encoded.shape[1], activation='softmax')  # Output layer
])

model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train_encoded, epochs=20, batch_size=8, verbose=1)

loss, accuracy = model.evaluate(x_test, y_test_encoded, verbose=0)
print(f"\nTest Accuracy: {accuracy:.4f}")
y_pred = model.predict(x_test)
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test_encoded, axis=1)
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Epoch 1/20


c:\Users\Hashir\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3942 - loss: 1.0121  
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6635 - loss: 0.8607 
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7115 - loss: 0.7089 
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7115 - loss: 0.5749 
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7115 - loss: 0.4804 
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8558 - loss: 0.4183 
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8077 - loss: 0.3727 
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9423 - loss: 0.3336 
Epoch 9/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9423 - loss: 0.3061 
Epoch 10/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9423 - loss: 0.2799 
Epoch 11/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9712 - loss: 0.2482 
Epoch 12/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9615 - l